### Keras/TensorFlow Deep Neural Network for EMBER
This uses the exact same Funnel architecture and Out-of-Core Batching as the PyTorch version, but avoids the `[WinError 1114] c10.dll` crash.

In [1]:
import os
import time
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models, callbacks

print(f"TensorFlow version {tf.__version__} loaded.")

try:
    from thrember.features import PEFeatureExtractor
    ndim = PEFeatureExtractor.dim
    print(f"Features extractor imported. Dimension: {ndim}")
except ImportError:
    ndim = 2381
    print(f"Warning: 'thrember' not found. Using default dimension: {ndim}")


TensorFlow version 2.15.1 loaded.


In [2]:
# --- CONFIGURATION ---
DATASET_DIR = r"Z:\ember2024_train_data"
CHUNK_SIZE = 250000              
BATCH_SIZE = 4096                
EPOCHS = 10                      
# Lowered learning rate for stability
LEARNING_RATE = 0.0001

# Changed to v3 for the new architecture
checkpoint_path = os.path.join(DATASET_DIR, "ember_keras_dnn_v3.weights.h5")

In [3]:
# --- MODEL ARCHITECTURE ---
def build_malware_dnn(input_dim):
    model = models.Sequential([
        layers.InputLayer(input_shape=(input_dim,)),
        
        # INCREASED INITIAL CAPACTIY
        layers.Dense(4096),
        layers.BatchNormalization(),
        layers.Activation('relu'),
        layers.Dropout(0.4),
        
        layers.Dense(2048),
        layers.BatchNormalization(),
        layers.Activation('relu'),
        layers.Dropout(0.3),
        
        layers.Dense(1024),
        layers.BatchNormalization(),
        layers.Activation('relu'),
        layers.Dropout(0.3),
        
        layers.Dense(512),
        layers.BatchNormalization(),
        layers.Activation('relu'),
        layers.Dropout(0.2),
        
        layers.Dense(128),
        layers.BatchNormalization(),
        layers.Activation('relu'),
        layers.Dropout(0.1),
        
        layers.Dense(1, activation='sigmoid') # Keras uses explicit sigmoid for binary
    ])
    
    # ADDED CLIPVALUE TO PREVENT GRADIENT EXPLOSION
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=LEARNING_RATE, clipvalue=1.0),
        loss='binary_crossentropy',
        metrics=['accuracy', tf.keras.metrics.AUC(name='auc')]
    )
    return model

model = build_malware_dnn(ndim)
model.summary()


Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 batch_normalization (Batch  (None, 2381)              9524      
 Normalization)                                                  
                                                                 
 dense (Dense)               (None, 4096)              9756672   
                                                                 
 batch_normalization_1 (Bat  (None, 4096)              16384     
 chNormalization)                                                
                                                                 
 leaky_re_lu (LeakyReLU)     (None, 4096)              0         
                                                                 
 dropout (Dropout)           (None, 4096)              0         
                                                                 
 dense_1 (Dense)             (None, 2048)              

In [4]:
# --- DATASET SIZE CALCULATION (NO MEMMAP) ---
X_path = os.path.join(DATASET_DIR, "X_train.dat")
y_path = os.path.join(DATASET_DIR, "y_train.dat")

file_size = os.path.getsize(X_path)
nrows = file_size // (ndim * 4)
train_nrows = int(nrows * 0.9)

print(f"Total Rows: {nrows:,} | Training on {train_nrows:,} samples.")

Total Rows: 5,664,483 | Training on 5,098,034 samples.


In [5]:
# --- CUSTOM GENERATOR FOR KERAS (DIRECT FILE I/O) ---
class DirectDataGenerator(tf.keras.utils.Sequence):
    def __init__(self, X_path, y_path, total_rows, ndim, batch_size):
        self.X_path = X_path
        self.y_path = y_path
        self.total_rows = total_rows
        self.ndim = ndim
        self.batch_size = batch_size
    
    def __len__(self):
        # Calculate total batches across all chunks
        return (self.total_rows + self.batch_size - 1) // self.batch_size
    
    def __getitem__(self, idx):
        start_idx = idx * self.batch_size
        end_idx = min((idx + 1) * self.batch_size, self.total_rows)
        rows = end_idx - start_idx
        
        # Calculate exact byte offsets (4 bytes per float32/int32)
        X_offset = start_idx * self.ndim * 4
        y_offset = start_idx * 4
        
        # Read exact batch bytes directly from disk without mmap
        with open(self.X_path, 'rb') as f_x:
            f_x.seek(X_offset)
            batch_x = np.frombuffer(f_x.read(rows * self.ndim * 4), dtype=np.float32).reshape(rows, self.ndim)
            
        with open(self.y_path, 'rb') as f_y:
            f_y.seek(y_offset)
            batch_y = np.frombuffer(f_y.read(rows * 4), dtype=np.int32)
        
        # Remove unlabeled (-1)
        valid = batch_y != -1
        return batch_x[valid], batch_y[valid]

train_gen = DirectDataGenerator(X_path, y_path, train_nrows, ndim, BATCH_SIZE)

In [ ]:
# --- SMART RESUME & TRAIN ---
if os.path.exists(checkpoint_path):
    print(f"Loading weights from {checkpoint_path}...")
    model.load_weights(checkpoint_path)
else:
    print("Starting fresh training.")

cp_callback = callbacks.ModelCheckpoint(
    filepath=checkpoint_path,
    save_weights_only=True,
    verbose=1
)

lr_callback = callbacks.ReduceLROnPlateau(
    monitor='loss', 
    factor=0.5, 
    patience=1, 
    verbose=1
)

print("\nStarting Keras Training...")
model.fit(
    train_gen,
    epochs=EPOCHS,
    callbacks=[cp_callback, lr_callback]
    # Removed workers=4 and use_multiprocessing=True to prevent Windows deadlocks
)

Starting fresh training.

Starting Keras Training...
Epoch 1/10


1245/1245 [==============================] - ETA: 0s - loss: 0.7094 - accuracy: 0.4999 - auc: 0.4999
Epoch 1: saving model to Z:\ember2024_train_data\ember_keras_dnn_v3.weights.h5
1245/1245 [==============================] - 6606s 5s/step - loss: 0.7094 - accuracy: 0.4999 - auc: 0.4999 - lr: 1.0000e-04
Epoch 2/10
1245/1245 [==============================] - ETA: 0s - loss: 0.6996 - accuracy: 0.5004 - auc: 0.5004
Epoch 2: saving model to Z:\ember2024_train_data\ember_keras_dnn_v3.weights.h5
1245/1245 [==============================] - 6606s 5s/step - loss: 0.6996 - accuracy: 0.5004 - auc: 0.5004 - lr: 1.0000e-04
Epoch 3/10
1245/1245 [==============================] - ETA: 0s - loss: 0.6967 - accuracy: 0.4999 - auc: 0.4997
Epoch 3: saving model to Z:\ember2024_train_data\ember_keras_dnn_v3.weights.h5
1245/1245 [==============================] - 6650s 5s/step - loss: 0.6967 - accuracy: 0.4999 - auc: 0.4997 - lr: 1.0000e-04
